In [40]:
import os

# os.getcwd() = "get current working directory"
# This tells you the folder Python is currently working from
print(os.getcwd())

c:\Users\FLEX\Desktop\kenya-economic-analysis


In [41]:
import os

# Show current working directory
print("Working directory:")
print(os.getcwd())
print()

# Show everything inside it
print("Contents of this folder:")
for item in os.listdir('.'):
    print(f"  {item}")

Working directory:
c:\Users\FLEX\Desktop\kenya-economic-analysis

Contents of this folder:
  app.ipynb
  data
  scripts
  visuals


In [42]:
import os

# Create the folder structure for the project
# exist_ok=True means: if the folder already exists, don't crash — just move on
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/cleaned', exist_ok=True)
os.makedirs('visuals', exist_ok=True)
os.makedirs('scripts', exist_ok=True)

print("Folders created:")
for item in os.listdir('.'):
    # os.path.isdir checks if something is a folder (not a file)
    if os.path.isdir(item):
        print(f"  📁 {item}")

Folders created:
  📁 data
  📁 scripts
  📁 visuals


In [43]:
print("Files in data/raw/:")
for f in os.listdir('data/raw'):
    print(f"  {f}")

Files in data/raw/:
  API_FP.CPI.TOTL.ZG_DS2_en_excel_v2_41.xls
  API_NY.GDP.MKTP.KD.ZG_DS2_en_excel_v2_29.xls
  API_NY.GDP.PCAP.CD_DS2_en_excel_v2_281.xls
  API_NY.GDP.PCAP.CN_DS2_en_excel_v2_6926.xls
  Chapter-2-Economic-Performance-Tables.xlsx
  Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx
  Chapter-9-Energy-Tables.xlsx
  Monthly-exchange-rate-end-period.csv
  Monthly-Exchange-rate-period-average.csv
  WEO_Data.xls


In [44]:
# Let's see the exact filenames so we know exactly what to type later
# when we reference them in code
print("Files in data/raw/:")
for f in sorted(os.listdir('data/raw')):
    # repr() shows the exact string including any hidden spaces or characters
    print(f"  {repr(f)}")

Files in data/raw/:
  'API_FP.CPI.TOTL.ZG_DS2_en_excel_v2_41.xls'
  'API_NY.GDP.MKTP.KD.ZG_DS2_en_excel_v2_29.xls'
  'API_NY.GDP.PCAP.CD_DS2_en_excel_v2_281.xls'
  'API_NY.GDP.PCAP.CN_DS2_en_excel_v2_6926.xls'
  'Chapter-2-Economic-Performance-Tables.xlsx'
  'Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx'
  'Chapter-9-Energy-Tables.xlsx'
  'Monthly-Exchange-rate-period-average.csv'
  'Monthly-exchange-rate-end-period.csv'
  'WEO_Data.xls'


In [45]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [46]:
# Read the file with header=None
# This means: don't try to guess which row is the header
# Show me every row exactly as it is in the file
df_raw = pd.read_excel(
    'data/raw/API_FP.CPI.TOTL.ZG_DS2_en_excel_v2_41.xls',
    sheet_name='Data',
    header=None
)

print(f"Shape: {df_raw.shape}")
print()
print(df_raw.head(8))

Shape: (270, 70)

                            0                             1   \
0                  Data Source  World Development Indicators   
1            Last Updated Date           2026-02-24 00:00:00   
2                          NaN                           NaN   
3                 Country Name                  Country Code   
4                        Aruba                           ABW   
5  Africa Eastern and Southern                           AFE   
6                  Afghanistan                           AFG   
7   Africa Western and Central                           AFW   

                                      2               3       4       5   \
0                                    NaN             NaN     NaN     NaN   
1                                    NaN             NaN     NaN     NaN   
2                                    NaN             NaN     NaN     NaN   
3                         Indicator Name  Indicator Code  1960.0  1961.0   
4  Inflation, consumer pr

In [47]:
# This time we tell pandas: row 3 is the header
# skiprows=[0,1,2] skips the three metadata rows at the top
df = pd.read_excel(
    'data/raw/API_FP.CPI.TOTL.ZG_DS2_en_excel_v2_41.xls',
    sheet_name='Data',
    header=3,         # row index 3 becomes the column names  
)

# Now look at the column names
print("Column names (first 6):")
print(df.columns[:6].tolist())
print()

# Find Kenya
kenya = df[df['Country Code'] == 'KEN']
print(f"Kenya row found: {len(kenya)} row(s)")
print()
print(kenya.iloc[:, :6])  # show first 6 columns only — the full row is 70 columns wide

Column names (first 6):
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961']

Kenya row found: 1 row(s)

    Country Name Country Code                         Indicator Name  \
121        Kenya          KEN  Inflation, consumer prices (annual %)   

     Indicator Code      1960      1961  
121  FP.CPI.TOTL.ZG  1.243781  2.457002  


In [48]:
# Define our analysis years as strings
# They must be strings because that's how they appear as column names
years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024']

# From the kenya row, select only those year columns
# .values[0] at the end because kenya is still a DataFrame (1 row)
# we want just the values as a simple list, not a DataFrame
kenya_inflation = kenya[years].values[0]

print("Kenya inflation 2018-2024:")
for year, value in zip(years, kenya_inflation):
    print(f"  {year}: {value}")

Kenya inflation 2018-2024:
  2018: 4.689806467763299
  2019: 5.239637957647923
  2020: 5.40516207933392
  2021: 6.107936036591763
  2022: 7.659862682722239
  2023: 7.671396340294023
  2024: 4.489788542434479


In [49]:
def read_wb_indicator(filename, indicator_name):
    """
    Reads a World Bank XLS file and extracts Kenya's values for 2018-2024.
    
    filename       : just the filename, e.g. 'API_FP.CPI.TOTL.ZG_DS2...xls'
    indicator_name : what we want to call this column, e.g. 'inflation_pct'
    
    Returns a small DataFrame with columns: year, indicator_name
    """
    
    # Build the full path — always store files in data/raw/
    filepath = f'data/raw/{filename}'
    
    # Read the file — header on row 3 as we just learned
    df = pd.read_excel(filepath, sheet_name='Data', header=3)
    
    # Filter to Kenya only
    kenya = df[df['Country Code'] == 'KEN']
    
    # The years we care about — as strings to match column names
    years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024']
    
    # Build a clean two-column DataFrame: year + the indicator value
    # We loop through years, grab the value for each, store as a dict
    records = []
    for year in years:
        value = kenya[year].values[0]
        records.append({
            'year': int(year),       # store year as integer, not string
            indicator_name: value    # column name is whatever we passed in
        })
    
    return pd.DataFrame(records)


# Test it on the inflation file
df_inflation = read_wb_indicator(
    'API_FP.CPI.TOTL.ZG_DS2_en_excel_v2_41.xls',
    'inflation_pct'
)

print(df_inflation)
print()
print(f"Shape: {df_inflation.shape}")
print(f"Data types:\n{df_inflation.dtypes}")

   year  inflation_pct
0  2018       4.689806
1  2019       5.239638
2  2020       5.405162
3  2021       6.107936
4  2022       7.659863
5  2023       7.671396
6  2024       4.489789

Shape: (7, 2)
Data types:
year               int64
inflation_pct    float64
dtype: object


In [50]:
# Call the function 4 times — one per World Bank file
df_inflation = read_wb_indicator(
    'API_FP.CPI.TOTL.ZG_DS2_en_excel_v2_41.xls',
    'inflation_pct'
)

df_gdp_usd = read_wb_indicator(
    'API_NY.GDP.PCAP.CD_DS2_en_excel_v2_281.xls',
    'gdp_per_capita_usd'
)

df_gdp_kes = read_wb_indicator(
    'API_NY.GDP.PCAP.CN_DS2_en_excel_v2_6926.xls',
    'gdp_per_capita_kes'
)

df_gdp_growth = read_wb_indicator(
    'API_NY.GDP.MKTP.KD.ZG_DS2_en_excel_v2_29.xls',
    'gdp_growth_pct'
)

# Now merge all 4 into one table
# We use reduce() to merge them one by one on the 'year' column
# Think of it as: start with df1, merge df2 onto it, then merge df3, then df4
from functools import reduce

wb_clean = reduce(
    lambda left, right: pd.merge(left, right, on='year', how='left'),
    [df_inflation, df_gdp_usd, df_gdp_kes, df_gdp_growth]
)

print(wb_clean)
print()
print(f"Shape: {wb_clean.shape}")
print(f"\nAny missing values?")
print(wb_clean.isnull().sum())

   year  inflation_pct  gdp_per_capita_usd  gdp_per_capita_kes  gdp_growth_pct
0  2018       4.689806         1836.452755       186035.554688        5.647946
1  2019       5.239638         1960.408089       199944.565561        5.114159
2  2020       5.405162         1927.664590       205201.399214       -0.272766
3  2021       6.107936         2061.356221       226002.451823        7.590489
4  2022       7.659863         2109.562885       248645.715814        4.859981
5  2023       7.671396         1942.588028       271663.911256        5.719992
6  2024       4.489789         2132.434521       287500.116953        4.724740

Shape: (7, 5)

Any missing values?
year                  0
inflation_pct         0
gdp_per_capita_usd    0
gdp_per_capita_kes    0
gdp_growth_pct        0
dtype: int64


In [51]:
# Sort by year just to be safe — calculations below depend on order
wb_clean = wb_clean.sort_values('year').reset_index(drop=True)

# Build a cumulative price index with 2018 as the base year (= 100)
# Each year's index = previous year's index × (1 + inflation rate/100)
#
# Example:
#   2018: 100.0  (base)
#   2019: 100.0 × (1 + 5.24/100) = 105.24
#   2020: 105.24 × (1 + 5.41/100) = 110.93
#   ...and so on
#
# This tells us: prices that cost 100 KES in 2018
# cost 105.24 KES in 2019, 110.93 KES in 2020, etc.

price_index = [100.0]  # start at 100 in 2018

for i in range(1, len(wb_clean)):
    previous_index = price_index[i - 1]
    current_inflation = wb_clean.loc[i, 'inflation_pct']
    new_index = previous_index * (1 + current_inflation / 100)
    price_index.append(new_index)

wb_clean['cumulative_price_index'] = price_index

# Now convert nominal GDP per capita to REAL GDP per capita
# Real = Nominal / Price Index × 100
# This strips out inflation and shows purchasing power in 2018 KES
wb_clean['gdp_per_capita_kes_real'] = (
    wb_clean['gdp_per_capita_kes'] / wb_clean['cumulative_price_index'] * 100
)

# Show the result — this is the key comparison
print(wb_clean[['year', 'gdp_per_capita_kes', 'cumulative_price_index', 'gdp_per_capita_kes_real']])

   year  gdp_per_capita_kes  cumulative_price_index  gdp_per_capita_kes_real
0  2018       186035.554688              100.000000            186035.554688
1  2019       199944.565561              105.239638            189989.788488
2  2020       205201.399214              110.928011            184986.098133
3  2021       226002.451823              117.703423            192010.093014
4  2022       248645.715814              126.719343            196217.648367
5  2023       271663.911256              136.440487            199107.990669
6  2024       287500.116953              142.566376            201660.535413


In [52]:
# Growth percentage vs 2018 base
# Formula: (current value - base value) / base value × 100
# This tells us: compared to 2018, how much better or worse off is someone?

base_nominal = wb_clean.loc[0, 'gdp_per_capita_kes']
base_real    = wb_clean.loc[0, 'gdp_per_capita_kes_real']

wb_clean['nominal_growth_vs_2018_pct'] = (
    (wb_clean['gdp_per_capita_kes'] - base_nominal) / base_nominal * 100
)

wb_clean['real_growth_vs_2018_pct'] = (
    (wb_clean['gdp_per_capita_kes_real'] - base_real) / base_real * 100
)

# Also add purchasing power of KES 1,000 in 2018 terms
# If prices are 42% higher, your 1,000 KES only buys what 701 KES bought in 2018
wb_clean['purchasing_power_of_1000kes'] = (
    1000 / wb_clean['cumulative_price_index'] * 100
)

# Show the final World Bank clean table
print(wb_clean[['year', 'nominal_growth_vs_2018_pct', 
                'real_growth_vs_2018_pct', 
                'purchasing_power_of_1000kes']].round(1))

   year  nominal_growth_vs_2018_pct  real_growth_vs_2018_pct  \
0  2018                         0.0                      0.0   
1  2019                         7.5                      2.1   
2  2020                        10.3                     -0.6   
3  2021                        21.5                      3.2   
4  2022                        33.7                      5.5   
5  2023                        46.0                      7.0   
6  2024                        54.5                      8.4   

   purchasing_power_of_1000kes  
0                       1000.0  
1                        950.2  
2                        901.5  
3                        849.6  
4                        789.1  
5                        732.9  
6                        701.4  


KES 1,000 in 2018 had the purchasing power of only KES 701 by 2024. In six years, the shilling lost nearly 30% of its value to inflation alone. That's before we even look at the exchange rate depreciation against the dollar.
And look at 2020 — real growth was -0.6%. The average Kenyan was actually poorer in real terms than in 2018. COVID didn't just slow growth, it reversed it.
These three columns — nominal growth, real growth, purchasing power — are your three strongest chart ideas for the Power BI dashboard.

In [53]:
# Save to data/cleaned/
# index=False means don't write the row numbers (0,1,2,3...) as a column
wb_clean.to_csv('data/cleaned/wb_macro.csv', index=False)

# Verify it saved correctly by reading it back and checking shape
verify = pd.read_csv('data/cleaned/wb_macro.csv')
print(f"Saved: {verify.shape[0]} rows × {verify.shape[1]} columns")
print(f"Columns: {verify.columns.tolist()}")

Saved: 7 rows × 10 columns
Columns: ['year', 'inflation_pct', 'gdp_per_capita_usd', 'gdp_per_capita_kes', 'gdp_growth_pct', 'cumulative_price_index', 'gdp_per_capita_kes_real', 'nominal_growth_vs_2018_pct', 'real_growth_vs_2018_pct', 'purchasing_power_of_1000kes']


In [54]:
# WEO_Data.xls is actually a TSV file (tab-separated values)
# disguised with a .xls extension — the IMF does this
# So we DON'T use read_excel — we use read_csv with sep='\t'
# sep='\t' means: columns are separated by tab characters, not commas

weo_raw = pd.read_csv(
    'data/raw/WEO_Data.xls',
    sep='\t',          # tab separated
    encoding='utf-8'   # character encoding — handles special characters
)

print(f"Shape: {weo_raw.shape}")
print()
print("Column names:")
print(weo_raw.columns.tolist())
print()
print("First 3 rows, first 10 columns:")
print(weo_raw.iloc[:3, :10])

Shape: (40, 36)

Column names:
['WEO Country Code', 'WEO Subject Code', 'Country', 'Subject Descriptor', 'Units', 'Scale', 'Country/Series-specific Notes', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', '2027', 'Estimates Start After']

First 3 rows, first 10 columns:
  WEO Country Code WEO Subject Code Country  \
0              664           NGDP_R   Kenya   
1              664        NGDP_RPCH   Kenya   
2              664             NGDP   Kenya   

                        Subject Descriptor              Units     Scale  \
0  Gross domestic product, constant prices  National currency  Billions   
1  Gross domestic product, constant prices     Percent change     Units   
2   Gross domestic product, current prices  National currency  Billions   

                       Country/Series-specific Notes       2000       20

In [55]:
# WEO_Data.xls is actually a TSV file (tab-separated values)
# disguised with a .xls extension — the IMF does this
# So we DON'T use read_excel — we use read_csv with sep='\t'
# sep='\t' means: columns are separated by tab characters, not commas

weo_raw = pd.read_csv(
    'data/raw/WEO_Data.xls',
    sep='\t',          # tab separated
    encoding='utf-8'   # character encoding — handles special characters
)

print(f"Shape: {weo_raw.shape}")
print()
print("Column names:")
print(weo_raw.columns.tolist())
print()
print("First 3 rows, first 10 columns:")
print(weo_raw.iloc[:3, :10])

Shape: (40, 36)

Column names:
['WEO Country Code', 'WEO Subject Code', 'Country', 'Subject Descriptor', 'Units', 'Scale', 'Country/Series-specific Notes', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', '2027', 'Estimates Start After']

First 3 rows, first 10 columns:
  WEO Country Code WEO Subject Code Country  \
0              664           NGDP_R   Kenya   
1              664        NGDP_RPCH   Kenya   
2              664             NGDP   Kenya   

                        Subject Descriptor              Units     Scale  \
0  Gross domestic product, constant prices  National currency  Billions   
1  Gross domestic product, constant prices     Percent change     Units   
2   Gross domestic product, current prices  National currency  Billions   

                       Country/Series-specific Notes       2000       20

In [56]:
indicators_to_extract = {
    'PCPIPCH':     'inflation_pct_imf',
    'NGSD_NGDP':   'national_savings_pct_gdp',
    'GGXWDG_NGDP': 'govt_debt_pct_gdp',
    'GGXCNL_NGDP': 'govt_deficit_pct_gdp',
    'BCA_NGDPD':   'current_account_pct_gdp',
}

years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024']

records = []

for year in years:
    row_data = {'year': int(year)}
    
    for code, col_name in indicators_to_extract.items():
        match = weo_raw[weo_raw['WEO Subject Code'] == code]
        raw_value = match.iloc[0][year]
        
        try:
            clean_value = float(str(raw_value).replace(',', '').strip())
        except ValueError:
            clean_value = np.nan
            
        row_data[col_name] = clean_value
    
    records.append(row_data)

imf_clean = pd.DataFrame(records)

print(imf_clean)
print()
print("Missing values:")
print(imf_clean.isnull().sum())

   year  inflation_pct_imf  national_savings_pct_gdp  govt_debt_pct_gdp  \
0  2018              4.690                    13.965             56.449   
1  2019              5.240                    14.101             59.085   
2  2020              5.290                    14.905             67.965   
3  2021              6.109                    15.163             68.232   
4  2022              7.648                    14.029             67.800   
5  2023              7.675                    12.387             73.107   
6  2024              5.118                    12.571             69.873   

   govt_deficit_pct_gdp  current_account_pct_gdp  
0                -6.936                   -5.411  
1                -7.415                   -5.241  
2                -8.134                   -4.748  
3                -7.196                   -5.228  
4                -6.057                   -5.025  
5                -5.759                   -3.970  
6                -5.037                   

govt_debt_pct_gdp went from 56% in 2018 to 73% in 2023. Kenya was borrowing heavily while ordinary people were experiencing 7.7% inflation. That's a direct connection — government borrowing puts pressure on the shilling, which raises import costs, which raises the cost of living.
national_savings_pct_gdp dropped from 14.9% in 2020 to 12.4% in 2023. As real wages fell, people saved less. That's exactly what economic theory predicts and your data confirms it.
govt_deficit_pct_gdp is negative every single year — Kenya spent more than it earned every year from 2018 to 2024. Peak deficit was -8.1% in 2020, COVID year.

In [57]:
imf_clean.to_csv('data/cleaned/imf_macro.csv', index=False)

verify = pd.read_csv('data/cleaned/imf_macro.csv')
print(f"Saved: {verify.shape[0]} rows × {verify.shape[1]} columns")
print(f"Columns: {verify.columns.tolist()}")

Saved: 7 rows × 6 columns
Columns: ['year', 'inflation_pct_imf', 'national_savings_pct_gdp', 'govt_debt_pct_gdp', 'govt_deficit_pct_gdp', 'current_account_pct_gdp']


File 3 — KNBS Chapter 3 

In [58]:
# ExcelFile lets us inspect the file before reading any sheet
# This shows us all the sheet names inside the file
xl = pd.ExcelFile('data/raw/Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx')

print("Sheets in Chapter 3:")
for i, name in enumerate(xl.sheet_names):
    print(f"  {i:2d}. {name}")

Sheets in Chapter 3:
   0. Table 3.1
   1. Table 3.2
   2. Table 3.3
   3. Table 3.4
   4. Table 3.5 
   5. Table 3.6
   6. Table 3.7
   7. Table 3.8
   8. Table 3.9
   9. Table 3.10
  10. Table 3.11a
  11. Table 3.11b
  12. Table 3.12
  13. Table 3.13
  14. Table 3.14
  15. Table 3.15a
  16. Table 3.15b
  17. Table 3.15c
  18. Table 3.16
  19. Table 3.17
  20. Table 3.18
  21. Table 3.19
  22. Table 3.20
  23. Table 3.21
  24. Table 3.22


In [59]:
# Read Table 3.7 with header=None so we see every row as-is
# nrows=25 — only show first 25 rows, enough to understand the structure
df_37_raw = pd.read_excel(
    'data/raw/Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx',
    sheet_name='Table 3.7',
    header=None,
    nrows=25
)

# Print every row with its index number
# so we can see exactly where the headers and data start
for i, row in df_37_raw.iterrows():
    print(f"Row {i:2d}: {row.tolist()}")

Row  0: ['Table 3.7: Annual Average Wage Earnings1 per Employee, 2019- 2023', nan, nan, nan, nan, nan, nan]
Row  1: [nan, nan, nan, nan, nan, nan, 'KSh']
Row  2: [nan, 'Industry', 2019, 2020, 2021, 2022, '2023*']
Row  3: ['PRIVATE SECTOR:', nan, nan, nan, nan, nan, nan]
Row  4: [nan, 'Agriculture, forestry and fishing  ', 369443.3, 376329.8, 377630, 390422.2, 405478]
Row  5: [nan, 'Mining and quarrying', 631213, 649873.2, 666150.1, 703234, 747702.7]
Row  6: [nan, 'Manufacturing', 529912.1, 558307.6, 570371.5, 617660.2, 641731.6]
Row  7: [nan, 'Electricity, gas, steam and air conditioning supply', 1899820, 2052500, 2077242.5, 2206161.6, 2311934.3]
Row  8: [nan, 'Water supply; sewerage, waste management and remediation activities', 282240, 290473.3, 294899.2, 310717.4, 330909.3]
Row  9: [nan, 'Construction', 717755.7, 758453.2, 764324.7, 793421.2, 833123]
Row 10: [nan, 'Wholesale and retail trade; repair of motor vehicles and motorcycles', 821477.7, 874784.6, 892517.3, 942160.6, 1007700.

In [60]:
# Read the full table this time — no nrows limit
df_37_full = pd.read_excel(
    'data/raw/Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx',
    sheet_name='Table 3.7',
    header=None
)

print(f"Total rows: {len(df_37_full)}")
print()

# Print from row 24 onwards to find where PUBLIC SECTOR starts
# and where the table ends
for i, row in df_37_full.iterrows():
    if i >= 24:
        print(f"Row {i:2d}: {row.tolist()}")

Total rows: 61

Row 24: [nan, 'Activities of extraterritorial organizations and bodies', 3519080, 3699960, 3757013.2, 4009007, 4002034.7]
Row 25: [nan, 'TOTAL   PRIVATE SECTOR              ', 780072.9, 811030.1, 829084.2, 874682.7, 916307.6]
Row 26: ['PUBLIC  SECTOR:', nan, nan, nan, nan, nan, nan]
Row 27: [nan, 'Agriculture, forestry and fishing  ', 479060.9, 511470.3, 515093.2, 524004, 512012.5]
Row 28: [nan, 'Mining and quarrying', 479654.5, 519280.4, 522339.7, 540469.7, 544199.1]
Row 29: [nan, 'Manufacturing', 986737.4, 1048868.9, 1041861.6, 1069384.7, 1055646.9]
Row 30: [nan, 'Electricity, gas, steam and air conditioning supply', 1457147.2, 1518954.8, 1529985, 1558086.8, 1501474.1]
Row 31: [nan, 'Water supply; sewerage, waste management and remediation activities', 652965.2, 669139.8, 669044.7, 684467.8, 652613.1]
Row 32: [nan, 'Construction', 810870, 864611.3, 874117.1, 893062.1, 872650.5]
Row 33: [nan, 'Wholesale and retail trade; repair of motor vehicles and motorcycles', 99074

In [61]:
import re

def clean_label(text):
    """
    Cleans a KNBS industry label.
    Two problems to fix:
      1. Trailing whitespace: 'Agriculture, forestry and fishing  '
      2. Dotted leaders: 'Ministries .. .. .. .. .. .. .. ..'
    
    str.strip() removes leading and trailing whitespace
    re.sub() replaces a pattern with something else
      r'[.\s]+$' means: one or more dots or spaces at the end of the string
      '$' means: end of string
    """
    if pd.isna(text):
        return None
    text = str(text).strip()
    text = re.sub(r'[.\s]+$', '', text)
    return text.strip()


# Test it on the problematic labels we saw
test_labels = [
    'Agriculture, forestry and fishing  ',
    'Information and communication ',
    'Ministries and other extra-budgetary institutions    ..   ..   ..   ..   ..',
    'TOTAL   PRIVATE SECTOR              ',
]

print("Before → After:")
for label in test_labels:
    print(f"  '{label}'")
    print(f"  → '{clean_label(label)}'")
    print()
    

Before → After:
  'Agriculture, forestry and fishing  '
  → 'Agriculture, forestry and fishing'

  'Information and communication '
  → 'Information and communication'

  'Ministries and other extra-budgetary institutions    ..   ..   ..   ..   ..'
  → 'Ministries and other extra-budgetary institutions'

  'TOTAL   PRIVATE SECTOR              '
  → 'TOTAL   PRIVATE SECTOR'



In [62]:
def read_knbs_wage_table(sheet_name, wage_type):
    """
    Reads one KNBS wage table (3.7 nominal or 3.9 real).
    
    sheet_name : 'Table 3.7' or 'Table 3.9'
    wage_type  : 'nominal' or 'real' — we'll use this to label the data
    
    Returns a clean DataFrame with columns:
    year, sector, industry, wage_type, annual_wage_kes
    """
    
    df = pd.read_excel(
        'data/raw/Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx',
        sheet_name=sheet_name,
        header=None
    )
    
    # Row 2 contains: NaN, 'Industry', 2019, 2020, 2021, 2022, '2023*'
    # We hardcode the years because we know the structure
    years = [2019, 2020, 2021, 2022, 2023]
    
    # We'll track which sector we're in as we loop through rows
    # It starts as None and changes when we hit 'PRIVATE SECTOR:' or 'PUBLIC SECTOR:'
    current_sector = None
    
    records = []
    
    # Start at row 3 — skip title (0), unit (1), header (2)
    for i in range(3, len(df)):
        row = df.iloc[i]
        
        # Column 0 contains sector headers like 'PRIVATE SECTOR:'
        # Column 1 contains industry names
        col0 = str(row.iloc[0]) if pd.notna(row.iloc[0]) else ''
        col1 = row.iloc[1]
        
        # ── Detect sector header rows ──────────────────────────────
        # If column 0 contains 'PRIVATE' or 'PUBLIC', update current_sector
        # 'continue' skips to the next row — no data to extract here
        if 'PRIVATE' in col0.upper():
            current_sector = 'Private'
            continue
        if 'PUBLIC' in col0.upper():
            current_sector = 'Public'
            continue
            
        # ── Skip rows with no industry label ──────────────────────
        if pd.isna(col1):
            continue
        
        # Clean the industry label
        industry = clean_label(col1)
        
        # ── Skip total rows and footnote rows ─────────────────────
        # Total rows contain 'TOTAL' in the label
        # Footnote rows start with numbers like '1', '2', '3' or '*'
        if 'TOTAL' in industry.upper():
            continue
        if industry.startswith(('1', '2', '3', '*', 'Source')):
            continue
        if 'MEMORANDUM' in industry.upper():
            continue
            
        # ── Extract values for each year ──────────────────────────
        # Values are in columns 2, 3, 4, 5, 6
        # zip pairs each year with its column index
        for year, col_idx in zip(years, range(2, 7)):
            raw_value = row.iloc[col_idx]
            
            # Convert to float
            # Dash values '  -    ' will fail float() — catch them as NaN
            try:
                value = float(str(raw_value).replace(',', '').strip())
            except ValueError:
                value = np.nan
            
            # Only store rows where we have a valid sector and value
            if current_sector is not None and not np.isnan(value):
                records.append({
                    'year':            year,
                    'sector':          current_sector,
                    'industry':        industry,
                    'wage_type':       wage_type,
                    'annual_wage_kes': value
                })
    
    return pd.DataFrame(records)


# Test on Table 3.7 — nominal wages
wages_nominal = read_knbs_wage_table('Table 3.7', 'nominal')

print(f"Shape: {wages_nominal.shape}")
print(f"\nSectors found: {wages_nominal['sector'].unique()}")
print(f"Years found: {sorted(wages_nominal['year'].unique())}")
print(f"Industries found: {wages_nominal['industry'].nunique()}")
print(f"\nFirst 10 rows:")
print(wages_nominal.head(10))

Shape: (205, 5)

Sectors found: ['Private' 'Public']
Years found: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Industries found: 26

First 10 rows:
   year   sector                           industry wage_type  annual_wage_kes
0  2019  Private  Agriculture, forestry and fishing   nominal         369443.3
1  2020  Private  Agriculture, forestry and fishing   nominal         376329.8
2  2021  Private  Agriculture, forestry and fishing   nominal         377630.0
3  2022  Private  Agriculture, forestry and fishing   nominal         390422.2
4  2023  Private  Agriculture, forestry and fishing   nominal         405478.0
5  2019  Private               Mining and quarrying   nominal         631213.0
6  2020  Private               Mining and quarrying   nominal         649873.2
7  2021  Private               Mining and quarrying   nominal         666150.1
8  2022  Private               Mining and quarrying   nominal         703234.0
9  2023  Private          

In [63]:
# Read real wages using the same function
wages_real = read_knbs_wage_table('Table 3.9', 'real')

print(f"Nominal rows: {len(wages_nominal)}")
print(f"Real rows:    {len(wages_real)}")

# Combine both into one table
# pd.concat stacks DataFrames vertically — adds rows on top of each other
# ignore_index=True resets row numbers from 0 instead of keeping original indices
wages_combined = pd.concat([wages_nominal, wages_real], ignore_index=True)

print(f"Combined rows: {len(wages_combined)}")
print(f"\nWage types: {wages_combined['wage_type'].unique()}")
print(f"\nSample — Agriculture private sector, both wage types:")
agri = wages_combined[
    (wages_combined['industry'] == 'Agriculture, forestry and fishing') &
    (wages_combined['sector'] == 'Private')
]
print(agri.to_string(index=False))

Nominal rows: 205
Real rows:    205
Combined rows: 410

Wage types: ['nominal' 'real']

Sample — Agriculture private sector, both wage types:
 year  sector                          industry wage_type  annual_wage_kes
 2019 Private Agriculture, forestry and fishing   nominal         369443.3
 2020 Private Agriculture, forestry and fishing   nominal         376329.8
 2021 Private Agriculture, forestry and fishing   nominal         377630.0
 2022 Private Agriculture, forestry and fishing   nominal         390422.2
 2023 Private Agriculture, forestry and fishing   nominal         405478.0


In [64]:
# Check what industries are in real wages
print("Industries in REAL wages:")
print(sorted(wages_real['industry'].unique()))
print()

# Check if agriculture appears at all in real wages
agri_real = wages_real[
    wages_real['industry'].str.contains('Agriculture', case=False, na=False)
]
print(f"Agriculture rows in real wages: {len(agri_real)}")
print(agri_real)

Industries in REAL wages:
['Accommodation and food service activities………………………………………………………', 'Activities of extraterritorial organizations and bodies…………………………………………', 'Activities of households as employers; undifferentiated goods- and services-producing activities of households for own use  …………', 'Administrative and support service activities……………………………………', 'Agriculture, forestry and fishing .. …………………………………………………………', 'Arts, entertainment and recreation………………………………………………………', 'Construction……………………………………………………………………………………', 'Corporations Controlled by the Government3', 'County governments', 'Education………………………………………………………………………………', 'Electricity, gas, steam and air conditioning supply……………………………………………………', 'Financial and insurance activities……………………………………………………………', 'Human health and social work activities…………………………………………………', 'Information and communication ………………………………………………………………', 'Manufacturing……………………………………………………………………………', 'Mining and quarrying…………………………………………………………………', 'Mini

In [65]:
def clean_label(text):
    """
    Updated version — handles three problems:
    1. Trailing whitespace
    2. Dots/ellipses at the end:  'Agriculture  ..'
    3. Dots/ellipses in the middle: 'Agriculture .. ………… and fishing'
    
    re.sub(r'\s*[.…]+\s*', ' ', text) means:
      \s*   — zero or more spaces
      [.…]+ — one or more dot or ellipsis characters
      \s*   — zero or more spaces after
      Replace all of that with a single space
    Then .strip() removes any leading/trailing spaces left over
    """
    if pd.isna(text):
        return None
    text = str(text).strip()
    
    # Remove dotted leaders anywhere in the string
    text = re.sub(r'\s*[.…]+\s*', ' ', text)
    
    # Clean up any double spaces that remain
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


# Test on the problematic labels from Table 3.9
test_labels = [
    'Agriculture, forestry and fishing .. …………………………………………………………',
    'Information and communication ………………………………………………………………',
    'Accommodation and food service activities………………………………………………………',
    'Ministries and other extra-budgetary institutions    ..   ..   ..   ..   ..',
    'Agriculture, forestry and fishing  ',   # original from 3.7 — must still work
]

print("Before → After:")
for label in test_labels:
    print(f"  IN : '{label}'")
    print(f"  OUT: '{clean_label(label)}'")
    print()

Before → After:
  IN : 'Agriculture, forestry and fishing .. …………………………………………………………'
  OUT: 'Agriculture, forestry and fishing'

  IN : 'Information and communication ………………………………………………………………'
  OUT: 'Information and communication'

  IN : 'Accommodation and food service activities………………………………………………………'
  OUT: 'Accommodation and food service activities'

  IN : 'Ministries and other extra-budgetary institutions    ..   ..   ..   ..   ..'
  OUT: 'Ministries and other extra-budgetary institutions'

  IN : 'Agriculture, forestry and fishing  '
  OUT: 'Agriculture, forestry and fishing'



In [66]:
# Re-run both tables with the fixed clean_label function
wages_nominal = read_knbs_wage_table('Table 3.7', 'nominal')
wages_real    = read_knbs_wage_table('Table 3.9', 'real')

# Combine
wages_combined = pd.concat([wages_nominal, wages_real], ignore_index=True)

# Now check agriculture again — should show both nominal and real
agri = wages_combined[
    (wages_combined['industry'] == 'Agriculture, forestry and fishing') &
    (wages_combined['sector'] == 'Private')
]
print("Agriculture, Private sector — nominal vs real:")
print(agri.to_string(index=False))
print()

# Check all industries match between nominal and real
nominal_industries = set(wages_nominal['industry'].unique())
real_industries    = set(wages_real['industry'].unique())

only_in_nominal = nominal_industries - real_industries
only_in_real    = real_industries - nominal_industries

print(f"Industries only in nominal: {only_in_nominal}")
print(f"Industries only in real:    {only_in_real}")

Agriculture, Private sector — nominal vs real:
 year  sector                          industry wage_type  annual_wage_kes
 2019 Private Agriculture, forestry and fishing   nominal         369443.3
 2020 Private Agriculture, forestry and fishing   nominal         376329.8
 2021 Private Agriculture, forestry and fishing   nominal         377630.0
 2022 Private Agriculture, forestry and fishing   nominal         390422.2
 2023 Private Agriculture, forestry and fishing   nominal         405478.0
 2019 Private Agriculture, forestry and fishing      real         356885.6
 2020 Private Agriculture, forestry and fishing      real         347598.7
 2021 Private Agriculture, forestry and fishing      real         328060.1
 2022 Private Agriculture, forestry and fishing      real         314299.0
 2023 Private Agriculture, forestry and fishing      real         302572.9

Industries only in nominal: set()
Industries only in real:    set()


In [67]:
# Add monthly equivalent — annual divided by 12
# More intuitive for readers than annual figures
wages_combined['monthly_wage_kes'] = wages_combined['annual_wage_kes'] / 12

# Quick summary before saving
print("=== WAGES SUMMARY ===")
print(f"Total rows: {len(wages_combined)}")
print(f"Sectors: {wages_combined['sector'].unique()}")
print(f"Years: {sorted(wages_combined['year'].unique())}")
print(f"Industries: {wages_combined['industry'].nunique()}")
print(f"Wage types: {wages_combined['wage_type'].unique()}")
print()

# Show average monthly wage across all private sector industries
# groupby splits data into groups, .mean() averages each group
private_avg = wages_combined[
    (wages_combined['sector'] == 'Private') &
    (wages_combined['wage_type'] == 'nominal')
].groupby('year')['monthly_wage_kes'].mean().round(0)

print("Average nominal monthly wage — Private sector (KES):")
print(private_avg.to_string())
print()

# Save
wages_combined.to_csv('data/cleaned/knbs_wages.csv', index=False)
verify = pd.read_csv('data/cleaned/knbs_wages.csv')
print(f"Saved: {verify.shape[0]} rows × {verify.shape[1]} columns")

=== WAGES SUMMARY ===
Total rows: 410
Sectors: ['Private' 'Public']
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Industries: 26
Wage types: ['nominal' 'real']

Average nominal monthly wage — Private sector (KES):
year
2019     87772.0
2020     92033.0
2021     93791.0
2022     98953.0
2023    103160.0

Saved: 410 rows × 6 columns


In [68]:
df_cpi_raw = pd.read_excel(
    'data/raw/Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx',
    sheet_name='Table 3.15a',
    header=None
)

print(f"Shape: {df_cpi_raw.shape}")
print()
for i, row in df_cpi_raw.iterrows():
    print(f"Row {i:2d}: {row.tolist()}")

Shape: (16, 8)

Row  0: ['Table 3.15a: Overall Consumer Price Indices and Rates of Inflation by COICOP Divisions, 2019 – 2023', nan, nan, nan, nan, nan, nan, nan]
Row  1: ['13 COICOP Divisions', 'Share (%)', 2019.0, 2020.0, 2021.0, 2022.0, 2023.0, '% Change 2023/2022_x000D_\n']
Row  2: ['Food and Non-Alcoholic Beverages…………………………………………………', 32.9094, 107.03, 116.73, 126.65, 143.26, 157.12, 9.7]
Row  3: ['Alcoholic Beverages, Tobacco and Narcotics……………………………………', 3.3289, 103.53, 109.98, 113.02, 118.28, 129.17, 9.2]
Row  4: ['Clothing and Footwear…………………………………..…………………………', 2.9914, 101.05, 103.5, 105.91, 108.39, 111.67, 3]
Row  5: ['Housing, Water, Electricity, Gas and Other Fuels……..…………………………', 14.6124, 101.0, 103.33, 108.26, 114.68, 123.99, 8.1]
Row  6: ['Furnishings, Household Equipment and Routine Household Maintenance…………', 3.7372, 101.08, 103.0, 107.19, 116.43, 122.8, 5.5]
Row  7: ['Health……………………………..………………………………………………………..', 2.9116, 100.56, 102.69, 106.2, 107.55, 110.11, 2.4]
Row

In [69]:
def read_cpi_categories():
    
    df = pd.read_excel(
        'data/raw/Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx',
        sheet_name='Table 3.15a',
        header=None
    )
    
    years = [2019, 2020, 2021, 2022, 2023]
    records = []
    
    # Data runs from row 2 to row 15 inclusive
    # Row 1 is the header, row 0 is the title
    for i in range(2, 16):
        row = df.iloc[i]
        
        # Column 0: category name — clean it
        category = clean_label(row.iloc[0])
        
        # Column 1: weight (how much of household budget this category represents)
        try:
            weight = float(str(row.iloc[1]).replace(',', '').strip())
        except ValueError:
            weight = np.nan
        
        # Columns 2-6: CPI index values for 2019-2023
        for year, col_idx in zip(years, range(2, 7)):
            try:
                cpi_value = float(str(row.iloc[col_idx]).replace(',', '').strip())
            except ValueError:
                cpi_value = np.nan
            
            records.append({
                'year':       year,
                'category':   category,
                'weight_pct': weight,
                'cpi_index':  cpi_value
            })
    
    return pd.DataFrame(records)


cpi_df = read_cpi_categories()

print(f"Shape: {cpi_df.shape}")
print(f"Categories: {cpi_df['category'].nunique()}")
print()
print("All categories and their weights (2019 values):")
snapshot = cpi_df[cpi_df['year'] == 2019][['category', 'weight_pct', 'cpi_index']]
print(snapshot.sort_values('weight_pct', ascending=False).to_string(index=False))

Shape: (70, 4)
Categories: 14

All categories and their weights (2019 values):
                                                             category  weight_pct  cpi_index
                                        Weighted Average of All Items    100.0000     103.16
                                     Food and Non-Alcoholic Beverages     32.9094     107.03
                     Housing, Water, Electricity, Gas and Other Fuels     14.6124     101.00
                                                            Transport      9.6468     102.40
                               Restaurants and Accommodation Services      8.0991     101.03
                                        Information and Communication      7.7840     100.49
                                                   Education Services      5.5620     100.19
Personal Care, Social Protection and Miscellaneous Goods and Services      4.4532     101.28
   Furnishings, Household Equipment and Routine Household Maintenance      3.7372   

70 rows, 14 categories, all weights visible. Now read that weight column — this is the economics insight, not just data:
Food alone is 32.9% of the average Kenyan household budget. Housing is 14.6%. Transport is 9.6%. Those three categories together are 57% of what a Kenyan household spends.
Now remember — food prices rose 46.8% and transport 48.2% between 2019 and 2023. The categories that eat the most of a poor household's budget are exactly the ones that rose fastest. That's why lower income groups experienced higher effective inflation than upper income groups — not because prices were different, but because their spending is concentrated in the most volatile categories.

In [70]:
# Calculate % price change vs 2019 base for each category
# Step 1: get the 2019 CPI value for each category
base_2019 = cpi_df[cpi_df['year'] == 2019][['category', 'cpi_index']].copy()
base_2019 = base_2019.rename(columns={'cpi_index': 'cpi_2019'})

# Step 2: merge base values back onto the full table
# Now every row knows what its category's 2019 value was
cpi_df = cpi_df.merge(base_2019, on='category', how='left')

# Step 3: calculate % change vs 2019
cpi_df['price_change_vs_2019_pct'] = (
    (cpi_df['cpi_index'] - cpi_df['cpi_2019']) / cpi_df['cpi_2019'] * 100
).round(1)

# Show the 2023 rankings — most to least expensive
print("Price increase 2019 → 2023 by category:")
print()
changes_2023 = cpi_df[cpi_df['year'] == 2023][
    ['category', 'weight_pct', 'price_change_vs_2019_pct']
].sort_values('price_change_vs_2019_pct', ascending=False)

for _, row in changes_2023.iterrows():
    bar = '█' * int(row['price_change_vs_2019_pct'] / 2)
    print(f"  {row['category'][:42]:42s} {row['price_change_vs_2019_pct']:+.1f}% {bar}")

# Save
cpi_df.to_csv('data/cleaned/knbs_cpi_categories.csv', index=False)
print()
print(f"Saved: {len(cpi_df)} rows × {len(cpi_df.columns)} columns")

Price increase 2019 → 2023 by category:

  Transport                                  +48.2% ████████████████████████
  Food and Non-Alcoholic Beverages           +46.8% ███████████████████████
  Weighted Average of All Items              +29.6% ██████████████
  Alcoholic Beverages, Tobacco and Narcotics +24.8% ████████████
  Housing, Water, Electricity, Gas and Other +22.8% ███████████
  Furnishings, Household Equipment and Routi +21.5% ██████████
  Personal Care, Social Protection and Misce +17.9% ████████
  Restaurants and Accommodation Services     +15.7% ███████
  Recreation, Sport and Culture              +12.9% ██████
  Clothing and Footwear                      +10.5% █████
  Health                                     +9.5% ████
  Education Services                         +6.8% ███
  Information and Communication              +5.5% ██
  Insurance and Financial Services           +4.3% ██

Saved: 70 rows × 6 columns


In [71]:
df_316_raw = pd.read_excel(
    'data/raw/Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx',
    sheet_name='Table 3.16',
    header=None
)

for i, row in df_316_raw.iterrows():
    print(f"Row {i:2d}: {row.tolist()}")

Row  0: ['Table 3.16: Annual Inflation, 2019- 2023', nan, nan, nan, nan, nan]
Row  1: [nan, nan, nan, nan, nan, 'Per cent']
Row  2: ['Income Group ', 2019.0, 2020.0, 2021.0, 2022.0, 2023]
Row  3: ['Nairobi Lower Income', 5.3, 6.2, 6.0, 8.1, 7.4]
Row  4: ['Nairobi Middle Income', 5.2, 2.6, 4.2, 6.0, 6.6]
Row  5: ['Nairobi Upper Income', 5.9, 2.6, 4.0, 5.7, 6.1]
Row  6: ['Nairobi ', 5.3, 4.7, 5.2, 7.2, 7]
Row  7: ['Rest of Urban Areas', 5.2, 5.9, 6.7, 8.0, 8.1]
Row  8: ['Overall Inflation ', 5.2, 5.4, 6.1, 7.7, 7.7]
Row  9: ['Notes:', nan, nan, nan, nan, nan]
Row 10: ['1. Nairobi Lower Income Group constitute of households that were spending KSh 46,355 or less per month in February 2016 (they constitute 70.89 per cent of all households in Nairobi). ', nan, nan, nan, nan, nan]
Row 11: ['2. Nairobi Middle Income Group consitute of households that were spending between KSh 46,356 up to and including KSh 184,394 per month in February 2016 (they constitute 25.58 per cent of all  households in

Lower income: spending KES 46,355 or less per month — 70.89% of Nairobi households.
Middle income: KES 46,356 to KES 184,394 — 25.58% of households.
Upper income: KES 184,395 or more — only 3.53% of households.
So when you say "lower income Kenyans experienced higher inflation" — you're talking about nearly 71% of Nairobi. That's not a fringe group. That's most people. Worth one sentence in your writeup.

In [72]:
def read_inflation_by_income():
    
    df = pd.read_excel(
        'data/raw/Chapter-3-Employment-Earnings-and-Consumer-Price-Indices-Tables.xlsx',
        sheet_name='Table 3.16',
        header=None
    )
    
    years = [2019, 2020, 2021, 2022, 2023]
    records = []
    
    # Data is rows 3-8, clean and simple
    for i in range(3, 9):
        row = df.iloc[i]
        
        # Clean the income group label
        group = clean_label(row.iloc[0])
        
        # Skip footnote rows — they start with 'Notes' or numbers
        if not group or group.startswith(('Notes', '1.', '2.', '3.')):
            continue
        
        for year, col_idx in zip(years, range(1, 6)):
            try:
                value = float(str(row.iloc[col_idx]).replace(',', '').strip())
            except ValueError:
                value = np.nan
            
            records.append({
                'year':          year,
                'income_group':  group,
                'inflation_pct': value
            })
    
    return pd.DataFrame(records)


income_inflation = read_inflation_by_income()

# Pivot to wide format for easy reading
# This flips years into columns so you can compare groups side by side
pivot = income_inflation.pivot(
    index='income_group',
    columns='year',
    values='inflation_pct'
)

print("Annual inflation by income group (%):")
print()
print(pivot.to_string())
print()

# The gap — lower minus upper each year
print("Inflation gap: Lower income MINUS Upper income (percentage points):")
lower = income_inflation[income_inflation['income_group'] == 'Nairobi Lower Income'].set_index('year')['inflation_pct']
upper = income_inflation[income_inflation['income_group'] == 'Nairobi Upper Income'].set_index('year')['inflation_pct']
gap = (lower - upper).round(1)
print(gap.to_string())

# Save
income_inflation.to_csv('data/cleaned/knbs_inflation_income.csv', index=False)
print()
print(f"Saved: {len(income_inflation)} rows × {len(income_inflation.columns)} columns")

Annual inflation by income group (%):

year                   2019  2020  2021  2022  2023
income_group                                       
Nairobi                 5.3   4.7   5.2   7.2   7.0
Nairobi Lower Income    5.3   6.2   6.0   8.1   7.4
Nairobi Middle Income   5.2   2.6   4.2   6.0   6.6
Nairobi Upper Income    5.9   2.6   4.0   5.7   6.1
Overall Inflation       5.2   5.4   6.1   7.7   7.7
Rest of Urban Areas     5.2   5.9   6.7   8.0   8.1

Inflation gap: Lower income MINUS Upper income (percentage points):
year
2019   -0.6
2020    3.6
2021    2.0
2022    2.4
2023    1.3

Saved: 30 rows × 3 columns


In [73]:
cbk_raw = pd.read_csv(
    'data/raw/Monthly-Exchange-rate-period-average.csv',
    header=None
)

print(f"Shape: {cbk_raw.shape}")
print()
print("First 6 rows:")
for i, row in cbk_raw.iterrows():
    if i < 6:
        print(f"Row {i}: {row.tolist()[:6]}")  # first 6 columns only
print()
print("Last 5 rows:")
print(cbk_raw.tail(5).iloc[:, :6].to_string())

Shape: (299, 32)

First 6 rows:
Row 0: [nan, 'Kenya Shilling Monthly Average Exchange Rates\\1', nan, nan, nan, nan]
Row 1: ['Year', 'Month', nan, 'United States dollar', 'Sterling pound', 'Euro']
Row 2: ['1993', '1', 'Jan-93', '36.23', '55.623', nan]
Row 3: ['1993', '2', 'Feb-93', '36.557', '52.675', nan]
Row 4: ['1993', '3', 'Mar-93', '43.121', '62.919', nan]
Row 5: ['1993', '4', 'Apr-93', '51.879', '80.336', nan]

Last 5 rows:
        0    1                                                  2        3        4        5
294  2017    5                                             May-17  103.262  133.456  114.049
295   NaN  NaN                                                NaN      NaN      NaN      NaN
296   NaN  NaN  \1 Unweighted average of buying and selling rates      NaN      NaN      NaN
297   NaN  NaN       \2 Implies currency units per Kenya Shilling      NaN      NaN      NaN
298   NaN  NaN                      Source: Central Bank of Kenya      NaN      NaN      NaN


In [74]:
def read_cbk_fx():
    
    df = pd.read_csv(
        'data/raw/Monthly-Exchange-rate-period-average.csv',
        header=None,
        skiprows=2      # skip title row and header row — we'll name columns ourselves
    )
    
    # Name the columns we care about
    # Column 0 = year, 1 = month number, 2 = month label, 3 = USD rate
    df.columns = ['year', 'month_num', 'month_label', 'usd_kes'] + \
                 [f'col_{i}' for i in range(4, len(df.columns))]
    
    # Keep only the 4 columns we need
    df = df[['year', 'month_num', 'month_label', 'usd_kes']].copy()
    
    # Forward-fill the year column
    # ffill() carries the last valid value forward to fill NaN gaps
    # This turns: 1993, NaN, NaN... into: 1993, 1993, 1993...
    df['year'] = df['year'].ffill()
    
    # Drop footnote rows — they have no month number
    df = df.dropna(subset=['month_num'])
    
    # Convert to numeric — everything came in as strings
    df['year']     = pd.to_numeric(df['year'],     errors='coerce')
    df['month_num'] = pd.to_numeric(df['month_num'], errors='coerce')
    df['usd_kes']  = pd.to_numeric(df['usd_kes'],  errors='coerce')
    
    # Drop any remaining bad rows
    df = df.dropna(subset=['year', 'usd_kes'])
    
    # Filter to 2010 onwards — earlier data not relevant to our project
    df = df[df['year'] >= 2010].copy()
    
    # Convert year to integer
    df['year'] = df['year'].astype(int)
    
    return df


cbk_monthly = read_cbk_fx()

print(f"Shape: {cbk_monthly.shape}")
print(f"Years: {sorted(cbk_monthly['year'].unique())}")
print()
print("Sample — first 3 and last 3 rows:")
print(pd.concat([cbk_monthly.head(3), cbk_monthly.tail(3)]).to_string(index=False))

Shape: (89, 4)
Years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]

Sample — first 3 and last 3 rows:
 year  month_num month_label  usd_kes
 2010        1.0      Jan-10   75.786
 2010        2.0      Feb-10   76.730
 2010        3.0      Mar-10   76.947
 2017        3.0      Mar-17  102.853
 2017        4.0      Apr-17  103.325
 2017        5.0      May-17  103.262


In [75]:
# Annual average KES/USD rate
cbk_annual = cbk_monthly.groupby('year')['usd_kes'].agg(
    annual_avg_usd_kes='mean',
    months_available='count'
).reset_index()

cbk_annual['annual_avg_usd_kes'] = cbk_annual['annual_avg_usd_kes'].round(2)

print("Annual average KES/USD rate (CBK data):")
print(cbk_annual.to_string(index=False))
print()
print("Note: 2017 only has 5 months of data")

# Save
cbk_annual.to_csv('data/cleaned/cbk_fx.csv', index=False)
print()
print(f"Saved: {len(cbk_annual)} rows × {len(cbk_annual.columns)} columns")

Annual average KES/USD rate (CBK data):
 year  annual_avg_usd_kes  months_available
 2010               79.23                12
 2011               88.81                12
 2012               84.53                12
 2013               86.12                12
 2014               87.92                12
 2015               98.18                12
 2016              101.50                12
 2017              103.36                 5

Note: 2017 only has 5 months of data

Saved: 8 rows × 3 columns


In [76]:
df_fuel_raw = pd.read_excel(
    'data/raw/Chapter-9-Energy-Tables.xlsx',
    sheet_name='Table 9.6',
    header=None
)

print(f"Shape: {df_fuel_raw.shape}")
print()
for i, row in df_fuel_raw.iterrows():
    print(f"Row {i:2d}: {row.tolist()}")

Shape: (42, 6)

Row  0: [nan, nan, nan, nan, nan, nan]
Row  1: ['Year', 'Month', 'KSh per Litre', nan, nan, 'KSh per \n13 Kg cylinder']
Row  2: [nan, nan, 'Motor Spirit Premium', nan, 'Illuminating Kerosene', 'Liqufied Petroleum Gas (LPG)']
Row  3: [nan, nan, nan, 'Light Diesel Oil', nan, nan]
Row  4: [2019, 'January', 104.99, 103.1, 102.56, 2193.13]
Row  5: [nan, 'March', 102.13, 97.47, 97.63, 2191.84]
Row  6: [nan, 'June', 115.82, 105.57, 105.48, 2192.25]
Row  7: [nan, 'September', 113.57, 103.9, 101.44, 2141.31]
Row  8: [nan, 'December', 109.91, 102.28, 102.81, 2101.16]
Row  9: [nan, 'Annual Average1', 109.63, 102.92, 102.29, 2161.26]
Row 10: [2020, 'January', 110.61, 102.81, 104.46, 2144.81]
Row 11: [nan, 'March', 112.07, 102.93, 96.72, 2065.98]
Row 12: [nan, 'June', 90.34, 75.88, 63.79, 2078.5]
Row 13: [nan, 'September', 106.3, 95.45, 84.09, 2033.57]
Row 14: [nan, 'December', 107.69, 92.75, 84.5, 1977.36]
Row 15: [nan, 'Annual Average1', 103.28, 93.96, 84.6, 2056.84]
Row 16: [2021

In [77]:
def read_fuel_prices():
    
    df = pd.read_excel(
        'data/raw/Chapter-9-Energy-Tables.xlsx',
        sheet_name='Table 9.6',
        header=None
    )
    
    # Forward-fill the year column — same pattern as CBK
    df[0] = df[0].ffill()
    
    records = []
    
    for i, row in df.iterrows():
        # We only want rows where month label contains 'Annual Average'
        month_label = str(row.iloc[1]).strip()
        if 'Annual Average' not in month_label:
            continue
        
        # Get the year
        try:
            year = int(float(str(row.iloc[0])))
        except ValueError:
            continue
        
        # Extract the four fuel values
        try:
            petrol   = float(str(row.iloc[2]).replace(',', ''))
            diesel   = float(str(row.iloc[3]).replace(',', ''))
            kerosene = float(str(row.iloc[4]).replace(',', ''))
            lpg_13kg = float(str(row.iloc[5]).replace(',', ''))
        except ValueError:
            continue
        
        records.append({
            'year':                  year,
            'petrol_kes_litre':      petrol,
            'diesel_kes_litre':      diesel,
            'kerosene_kes_litre':    kerosene,
            'lpg_13kg_kes':          lpg_13kg,
        })
    
    fuel_df = pd.DataFrame(records)
    
    # Add % change vs 2019 for each fuel type
    # We do this for all four columns at once using a loop
    base_year = fuel_df[fuel_df['year'] == 2019].iloc[0]
    
    for col in ['petrol_kes_litre', 'diesel_kes_litre', 
                'kerosene_kes_litre', 'lpg_13kg_kes']:
        base_val = base_year[col]
        fuel_df[f'{col}_change_vs_2019_pct'] = (
            (fuel_df[col] - base_val) / base_val * 100
        ).round(1)
    
    return fuel_df


fuel_df = read_fuel_prices()

print(fuel_df.to_string(index=False))
print()
fuel_df.to_csv('data/cleaned/energy_fuel_prices.csv', index=False)
print(f"Saved: {len(fuel_df)} rows × {len(fuel_df.columns)} columns")

 year  petrol_kes_litre  diesel_kes_litre  kerosene_kes_litre  lpg_13kg_kes  petrol_kes_litre_change_vs_2019_pct  diesel_kes_litre_change_vs_2019_pct  kerosene_kes_litre_change_vs_2019_pct  lpg_13kg_kes_change_vs_2019_pct
 2019            109.63            102.92              102.29       2161.26                                  0.0                                  0.0                                    0.0                              0.0
 2020            103.28             93.96               84.60       2056.84                                 -5.8                                 -8.7                                  -17.3                             -4.8
 2021            125.79            108.56              100.19       2280.91                                 14.7                                  5.5                                   -2.1                              5.5
 2022            157.34            139.69              127.38       2990.41                                 43.5

In [78]:
# Read back what was saved and inspect it
fuel_check = pd.read_csv('data/cleaned/energy_fuel_prices.csv')
print(fuel_check.to_string(index=False))

 year  petrol_kes_litre  diesel_kes_litre  kerosene_kes_litre  lpg_13kg_kes  petrol_kes_litre_change_vs_2019_pct  diesel_kes_litre_change_vs_2019_pct  kerosene_kes_litre_change_vs_2019_pct  lpg_13kg_kes_change_vs_2019_pct
 2019            109.63            102.92              102.29       2161.26                                  0.0                                  0.0                                    0.0                              0.0
 2020            103.28             93.96               84.60       2056.84                                 -5.8                                 -8.7                                  -17.3                             -4.8
 2021            125.79            108.56              100.19       2280.91                                 14.7                                  5.5                                   -2.1                              5.5
 2022            157.34            139.69              127.38       2990.41                                 43.5

In [79]:
import os

print("Files in data/cleaned/:")
for f in sorted(os.listdir('data/cleaned')):
    df_check = pd.read_csv(f'data/cleaned/{f}')
    print(f"  {f:40s} {df_check.shape[0]:3d} rows × {df_check.shape[1]:2d} cols")

Files in data/cleaned/:
  cbk_fx.csv                                 8 rows ×  3 cols
  energy_fuel_prices.csv                     5 rows ×  9 cols
  imf_macro.csv                              7 rows ×  6 cols
  knbs_cpi_categories.csv                   70 rows ×  6 cols
  knbs_inflation_income.csv                 30 rows ×  3 cols
  knbs_wages.csv                           410 rows ×  6 cols
  wb_macro.csv                               7 rows × 10 cols


In [80]:
# Load all cleaned files
wb        = pd.read_csv('data/cleaned/wb_macro.csv')
imf       = pd.read_csv('data/cleaned/imf_macro.csv')
cbk       = pd.read_csv('data/cleaned/cbk_fx.csv')
fuel      = pd.read_csv('data/cleaned/energy_fuel_prices.csv')
wages     = pd.read_csv('data/cleaned/knbs_wages.csv')
cpi_cats  = pd.read_csv('data/cleaned/knbs_cpi_categories.csv')
inc_infl  = pd.read_csv('data/cleaned/knbs_inflation_income.csv')

# ── Prepare wages summary ─────────────────────────────────────
# wages has 410 rows — one per industry/sector/year/type
# We need one row per year for the master table
# So we take the average across all private sector industries
# for nominal and real wages separately

wages_summary = wages[wages['sector'] == 'Private'].groupby(
    ['year', 'wage_type']
)['monthly_wage_kes'].mean().round(0).unstack('wage_type').reset_index()

# unstack() pivots wage_type values (nominal/real) into separate columns
# so we get: year | nominal | real
wages_summary.columns = ['year', 'avg_nominal_monthly_wage', 'avg_real_monthly_wage']

print("Wages summary:")
print(wages_summary.to_string(index=False))

Wages summary:
 year  avg_nominal_monthly_wage  avg_real_monthly_wage
 2019                   87772.0                84788.0
 2020                   92033.0                85007.0
 2021                   93791.0                81480.0
 2022                   98953.0                79660.0
 2023                  103160.0                76979.0


In [81]:
# ── Prepare CPI pivot ─────────────────────────────────────────
# cpi_cats has 70 rows — one per category per year
# We need one row per year with each category as its own column
# pivot_table does exactly this

cpi_pivot = cpi_cats.pivot_table(
    index='year',
    columns='category',
    values='cpi_index'
).reset_index()
cpi_pivot.columns.name = None  # removes the 'category' label above column names

# Rename long category names to short clean versions
cpi_pivot = cpi_pivot.rename(columns={
    'Food and Non-Alcoholic Beverages':                          'cpi_food',
    'Housing, Water, Electricity, Gas and Other Fuels':          'cpi_housing',
    'Transport':                                                  'cpi_transport',
    'Education Services':                                         'cpi_education',
    'Health':                                                     'cpi_health',
    'Information and Communication':                              'cpi_ict',
    'Clothing and Footwear':                                      'cpi_clothing',
    'Alcoholic Beverages, Tobacco and Narcotics':                 'cpi_alcohol_tobacco',
    'Furnishings, Household Equipment and Routine Household Maintenance': 'cpi_furnishings',
    'Restaurants and Accommodation Services':                     'cpi_restaurants',
    'Recreation, Sport and Culture':                              'cpi_recreation',
    'Insurance and Financial Services':                           'cpi_insurance',
    'Personal Care, Social Protection and Miscellaneous Goods and Services': 'cpi_personal_care',
    'Weighted Average of All Items':                              'cpi_overall'
})

# ── Prepare income inflation ──────────────────────────────────
# Extract just the three income groups we want as separate columns
lower = inc_infl[inc_infl['income_group'] == 'Nairobi Lower Income'][['year','inflation_pct']].rename(
    columns={'inflation_pct': 'inflation_lower_income'})

upper = inc_infl[inc_infl['income_group'] == 'Nairobi Upper Income'][['year','inflation_pct']].rename(
    columns={'inflation_pct': 'inflation_upper_income'})

overall = inc_infl[inc_infl['income_group'] == 'Overall Inflation'][['year','inflation_pct']].rename(
    columns={'inflation_pct': 'inflation_overall_knbs'})

print("CPI pivot shape:", cpi_pivot.shape)
print("CPI columns:", cpi_pivot.columns.tolist())
print()
print("Income inflation shapes:", lower.shape, upper.shape, overall.shape)

CPI pivot shape: (5, 15)
CPI columns: ['year', 'cpi_alcohol_tobacco', 'cpi_clothing', 'cpi_education', 'cpi_food', 'cpi_furnishings', 'cpi_health', 'cpi_housing', 'cpi_ict', 'cpi_insurance', 'cpi_personal_care', 'cpi_recreation', 'cpi_restaurants', 'cpi_transport', 'cpi_overall']

Income inflation shapes: (5, 2) (5, 2) (5, 2)


In [82]:
# ── Build master table ────────────────────────────────────────
# Start with all years 2018-2024 as the backbone
# Every other dataset merges onto this with how='left'
# so we never lose a year even if some files don't cover it

master = pd.DataFrame({'year': range(2018, 2025)})

# Merge each dataset one by one
# After each merge, print the shape so we can see columns being added
master = master.merge(wb,            on='year', how='left')
print(f"After World Bank:        {master.shape}")

master = master.merge(imf,           on='year', how='left')
print(f"After IMF:               {master.shape}")

master = master.merge(wages_summary, on='year', how='left')
print(f"After wages:             {master.shape}")

master = master.merge(cpi_pivot,     on='year', how='left')
print(f"After CPI categories:    {master.shape}")

master = master.merge(lower,         on='year', how='left')
master = master.merge(upper,         on='year', how='left')
master = master.merge(overall,       on='year', how='left')
print(f"After income inflation:  {master.shape}")

master = master.merge(fuel,          on='year', how='left')
print(f"After fuel prices:       {master.shape}")

After World Bank:        (7, 10)
After IMF:               (7, 15)
After wages:             (7, 17)
After CPI categories:    (7, 31)
After income inflation:  (7, 34)
After fuel prices:       (7, 42)


In [83]:
# ── Final derived columns ─────────────────────────────────────

# 1. Inflation gap: how much more the poor pay vs the wealthy
master['inflation_gap_lower_vs_upper'] = (
    master['inflation_lower_income'] - master['inflation_upper_income']
)

# 2. Real wage loss vs 2019
# How much purchasing power has the average worker lost since 2019?
base_real_wage = master.loc[master['year'] == 2019, 'avg_real_monthly_wage'].values[0]
master['real_wage_change_vs_2019_pct'] = (
    (master['avg_real_monthly_wage'] - base_real_wage) / base_real_wage * 100
).round(1)

# 3. How many litres of petrol can one month's wage buy?
# A concrete way to show purchasing power that anyone can relate to
master['litres_petrol_per_monthly_wage'] = (
    master['avg_nominal_monthly_wage'] / master['petrol_kes_litre']
).round(1)

# ── Data quality report ───────────────────────────────────────
print("=== MASTER TABLE — COMPLETENESS ===")
print(f"Shape: {master.shape}")
print()
print("Missing values per column:")
nulls = master.isnull().sum()
for col, count in nulls.items():
    # Show a bar: filled squares for populated, empty for missing
    filled = 7 - count
    bar = '█' * filled + '░' * count
    print(f"  {col:45s} [{bar}] {filled}/7")

=== MASTER TABLE — COMPLETENESS ===
Shape: (7, 45)

Missing values per column:
  year                                          [███████] 7/7
  inflation_pct                                 [███████] 7/7
  gdp_per_capita_usd                            [███████] 7/7
  gdp_per_capita_kes                            [███████] 7/7
  gdp_growth_pct                                [███████] 7/7
  cumulative_price_index                        [███████] 7/7
  gdp_per_capita_kes_real                       [███████] 7/7
  nominal_growth_vs_2018_pct                    [███████] 7/7
  real_growth_vs_2018_pct                       [███████] 7/7
  purchasing_power_of_1000kes                   [███████] 7/7
  inflation_pct_imf                             [███████] 7/7
  national_savings_pct_gdp                      [███████] 7/7
  govt_debt_pct_gdp                             [███████] 7/7
  govt_deficit_pct_gdp                          [███████] 7/7
  current_account_pct_gdp                       [████

In [85]:
master.to_csv('data/cleaned/master_annual.csv', index=False)

# Read it back and verify
verify = pd.read_csv('data/cleaned/master_annual.csv')
print(f"master_annual.csv saved")
print(f"   {verify.shape[0]} rows × {verify.shape[1]} columns")
print()

# Print the 5 most important columns as final confirmation
print("=== HEADLINE NUMBERS ===")
headline = verify[[
    'year',
    'inflation_pct',
    'gdp_per_capita_kes',
    'gdp_per_capita_kes_real',
    'purchasing_power_of_1000kes',
    'real_growth_vs_2018_pct'
]].round(1)
print(headline.to_string(index=False))

master_annual.csv saved
   7 rows × 45 columns

=== HEADLINE NUMBERS ===
 year  inflation_pct  gdp_per_capita_kes  gdp_per_capita_kes_real  purchasing_power_of_1000kes  real_growth_vs_2018_pct
 2018            4.7            186035.6                 186035.6                       1000.0                      0.0
 2019            5.2            199944.6                 189989.8                        950.2                      2.1
 2020            5.4            205201.4                 184986.1                        901.5                     -0.6
 2021            6.1            226002.5                 192010.1                        849.6                      3.2
 2022            7.7            248645.7                 196217.6                        789.1                      5.5
 2023            7.7            271663.9                 199108.0                        732.9                      7.0
 2024            4.5            287500.1                 201660.5                      

The story your data is already telling:

Nominal income rose 54.5% from 2018-2024. Real income rose only 8.4%
KES 1,000 in 2018 buys only KES 701 worth of goods in 2024
Real private sector wages fell 9.2% between 2019-2023
Transport +48%, food +47% — the categories the poor spend most on rose fastest
Lower income households paid 3.6 percentage points more inflation than upper income in 2020
Government debt hit 73% of GDP in 2023 while ordinary Kenyans got poorer in real terms